# PerturbationCatalogueLLM — End-to-End Pipeline

**Architecture:**
```
Perturb-seq CSV
     │
     ▼
[Step 1] Fine-tune PerturbationCatalogueLLM encoder  →  perturbation embeddings
         (pre-trained: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract)
     │
     ▼
[Step 2] Build FAISS RAG index   →  vectors + text documents
     │
     ▼
[Step 3] LLM Agent (Claude/GPT/Ollama)  ←  RAG retrieval
     │
     ▼
Grounded answers about perturbation effects
```

## 0. Setup

In [1]:
# Install dependencies
!pip install -q torch transformers scikit-learn faiss-cpu anthropic openai tqdm fastapi uvicorn pydantic

# Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/PerturbationCatalogue_LLM_2')
print('Working directory:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/PerturbationCatalogue_LLM_2


In [ ]:
# Set API keys (only needed for step 3)
import os
os.environ['ANTHROPIC_API_KEY'] = my_anthropic_api_key
# os.environ['OPENAI_API_KEY']  = 'your-key-here'   # for GPT-4

# Copy your data file into the expected location
import shutil, pathlib
pathlib.Path('data').mkdir(exist_ok=True)
shutil.copy(
    'perturb-seq_adamson_2016_pilot.csv',
    'data/perturb_seq.csv'
)
print('Data ready.')

Data ready.


## Step 1 — Fine-tune PerturbationCatalogueLLM encoder

In [3]:
project_root = '/content/drive/MyDrive/PerturbationCatalogue_LLM_2'

In [ ]:
import os

# Create all needed directories before calling train()
os.makedirs(f'{project_root}/embeddings/perturbation_catalogue_llm', exist_ok=True)
os.makedirs(f'{project_root}/models/perturbation_catalogue_llm', exist_ok=True)
os.makedirs(f'{project_root}/data/chroma_db', exist_ok=True)

print("Directories created:")
for d in ['embeddings/perturbation_catalogue_llm', 'models/perturbation_catalogue_llm', 'data/chroma_db']:
    print(f"  ✓ {project_root}/{d}")

In [ ]:
import sys
sys.path.insert(0, '.')

# Inline import from our script
from scripts.s01_finetune import train, parse_args
import argparse

# Configure training
args = argparse.Namespace(
    data_path   = 'data/perturb_seq.csv',
    output_dir  = 'embeddings/perturbation_catalogue_llm',
    epochs      = 100,      # More epochs because only 7 perturbations
    batch_size  = 4,
    lr          = 1e-3,
    embed_dim   = 256,
    max_genes   = 2048,
)

model, dataset, meta = train(args)
print('\nPerturbations embedded:', meta['perturbation_list'])

## Step 2 — Build RAG index

In [ ]:
from scripts.s02_build_rag_index import build_documents, PerturbationRAGIndex
import numpy as np, json

# Load embeddings from step 1
with open('embeddings/perturbation_catalogue_llm/embedding_index.json') as f:
    embedding_index = json.load(f)

pert_names = list(embedding_index.keys())
embeddings = np.array([embedding_index[p] for p in pert_names], dtype='float32')
print(f'Loaded {len(pert_names)} embeddings, dim={embeddings.shape[1]}')

# Build text documents
documents = build_documents('data/perturb_seq.csv', top_n=20)

# Align and index
doc_map  = {d['perturbation']: d for d in documents}
aligned_docs = [doc_map[n] for n in pert_names if n in doc_map]
aligned_embs = np.array([embedding_index[n] for n in pert_names if n in doc_map], dtype='float32')

rag_index = PerturbationRAGIndex()
rag_index.build(aligned_embs, aligned_docs)
rag_index.save('embeddings/rag_index')

print('\nRAG index built successfully.')

## Step 3 — Ask questions with the LLM agent

In [7]:
from scripts.s03_perturbation_agent import PerturbationAgent, get_backend, PerturbationRAGIndex

In [8]:
from scripts.s03_perturbation_agent import PerturbationAgent, get_backend, PerturbationRAGIndex

# Load the RAG index
rag = PerturbationRAGIndex.load('embeddings/rag_index')

# Pick your LLM backend: 'anthropic', 'openai', or 'ollama'
llm   = get_backend('anthropic')   # Uses ANTHROPIC_API_KEY
agent = PerturbationAgent(rag, llm, k_retrieve=3)

print('Agent ready. Perturbations available:', rag.get_all_perturbations())

✅ RAG index loaded: 7 perturbations from embeddings/rag_index
  LLM backend: Anthropic (claude-sonnet-4-5)
Agent ready. Perturbations available: ['BHLHE40', 'CREB1', 'DDIT3', 'EP300', 'SNAI1', 'SPI1', 'ZNF326']


In [9]:
# Example questions
questions = [
    'What happens when SPI1 is knocked out?',
    'Which genes does SNAI1 most strongly up-regulate?',
    'Compare the effects of EP300 and CREB1 knockouts.',
    'What biological pathway does DDIT3 appear to regulate based on its perturb-seq signature?',
]

for q in questions:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'{"="*60}')
    answer = agent.ask(q)
    print(answer)


Q: What happens when SPI1 is knocked out?
# SPI1 Knockout Effects

## Summary
Knockout of SPI1 results in substantial transcriptional remodeling with **369 significantly differentially expressed genes** (Padj < 0.05). The perturbation shows a balanced but distinct response with both up-regulated (20 genes) and down-regulated (20 genes among the top hits).

## Key Expression Changes

### Down-regulated Genes (Most Prominent)
The strongest effect is **down-regulation of PA2G4** (Log2FC=-1.49, Padj=3.49e-168), which encodes a proliferation-associated protein. Other notably suppressed genes include:

- **APOE** (Log2FC=-1.15): Apolipoprotein E, involved in lipid metabolism and immune function
- **LY6E** (Log2FC=-1.38): Interferon-induced immune regulator
- **IFITM2** (Log2FC=-2.14): Interferon-induced transmembrane protein with antiviral function
- **IFITM1** (Log2FC=-1.55): Related interferon response gene
- **CHI3L2** (Log2FC=-1.21): Chitinase-like protein involved in inflammation
- **M

In [ ]:
# Structured API — useful for automated analysis
result = agent.structured_query('SPI1', 'mechanism')
print(result['answer'])
print('\nMetadata:', result['metadata'])

In [ ]:
# Multi-turn conversation
agent.reset()

r1 = agent.ask('Tell me about the top down-regulated genes when ZNF326 is knocked out.')
print('Turn 1:', r1[:500], '...')

r2 = agent.ask('What do those genes have in common functionally?')
print('\nTurn 2 (follow-up):', r2[:500], '...')

## Optional — Start REST API server
Useful for connecting to a frontend or other tools.

```bash
python scripts/03_perturbation_agent.py --llm anthropic --serve --port 8000
```

Endpoints:
- `GET  /perturbations`       → list available genes
- `POST /ask`                 → free-text question
- `POST /structured`          → structured query
- `GET  /health`              → health check